# Week 78, Day 5: Automation project

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w078d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from decimal import Decimal, ROUND_HALF_UP

print(0.1 + 0.2 == 0.3, Decimal("0.1") + Decimal("0.2") == Decimal("0.3"))
pence = int(Decimal("-11.50") * 100)
print(pence)
in_gbp = Decimal(pence) / Decimal("1.15")
print(in_gbp)
print(int(Decimal("-869.5")), Decimal("-869.5").quantize(Decimal("1"), rounding=ROUND_HALF_UP))

These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('inbox/2026-10-bank.csv', 'w', encoding='utf-8') as f:
    f.write('date,description,amount,currency\n2026-10-03,TESCO STORES 2207,-23.40,GBP\n2026-10-05,Salary ACME LTD,2100.00,GBP\n2026-10-12,Cafe de Paris,-11.50,EUR\n2026-10-30,Rent October,-850.00,GBP\n')
with open('inbox/2026-11-card.csv', 'w', encoding='utf-8') as f:
    f.write('date,description,amount,currency\n2026-11-01,Amazon Marketplace,-25.00,USD\n2026-11-02,TESCO EXTRA,-41.10,GBP\n2026-11-02,Corner shop,abc,GBP\n2026-11-31,Bus,-2.00,GBP\n2026-11-04,Netflix,-10.99,GBP\n2026-11-05,Hotel,-120.00,eur\n')
with open('rates.html', 'w', encoding='utf-8') as f:
    f.write('<html><head><title>Daily rates</title></head><body>\n<h1>Exchange rates</h1>\n<table id="rates">\n  <tr><th>Currency</th><th>Units per GBP</th></tr>\n  <tr><td>EUR</td><td>1.1500</td></tr>\n  <tr><td>USD</td><td>1.2500</td></tr>\n  <tr><td>JPY</td><td>190.20</td></tr>\n</table>\n<p>Rates are indicative only.</p>\n</body></html>\n')

### Exercise: Step 1: reading statements

Write `read_statement(path)`. Read the CSV file with the `csv` module (`newline=""`, UTF-8). If the first row isn't exactly `["date", "description", "amount", "currency"]`, raise `ValueError(f"{NAME}: expected the header date,description,amount,currency")` (NAME is the file's name without its folder). Skip empty rows. For each other row, strip each field, then add an error string `NAME:LINE: PROBLEM` and skip the row, where PROBLEM is the first of: `expected 4 fields`; `bad date` (not accepted by `date.fromisoformat`); `bad amount` (not a valid `Decimal`, or with more than two decimal places); `bad currency` (not exactly three capital letters A to Z). Good rows become `{"date": date, "description": str, "pence": int, "currency": str}`. Return `(rows, errors)`.

In [ ]:
import csv
import re
from datetime import date
from decimal import Decimal, InvalidOperation
from pathlib import Path

HEADER = ["date", "description", "amount", "currency"]

def read_statement(path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_good_file():\n    \"the October statement\"\n    from datetime import date\n    rows, errors = read_statement(\"inbox/2026-10-bank.csv\")\n    assert errors == [] and len(rows) == 4\n    assert rows[2] == {\"date\": date(2026, 10, 12), \"description\": \"Cafe de Paris\", \"pence\": -1150, \"currency\": \"EUR\"}\n    assert [r[\"pence\"] for r in rows] == [-2340, 210000, -1150, -85000]\n\ndef test_problems():\n    \"every problem, with its line\"\n    rows, errors = read_statement(\"inbox/2026-11-card.csv\")\n    assert errors == [\"2026-11-card.csv:4: bad amount\", \"2026-11-card.csv:5: bad date\", \"2026-11-card.csv:7: bad currency\"]\n    assert [r[\"description\"] for r in rows] == [\"Amazon Marketplace\", \"TESCO EXTRA\", \"Netflix\"]\n    write_file(\"odd.csv\", \"date,description,amount,currency\\n\\n2026-01-01, Tea ,1.234,GBP\\n2026-01-02,x,1,GBP,extra\\n2026-01-03, Pie , 3.5 , GBP \\n\")\n    rows, errors = read_statement(\"odd.csv\")\n    assert errors == [\"odd.csv:3: bad amount\", \"odd.csv:4: expected 4 fields\"]\n    assert rows == [{\"date\": __import__(\"datetime\").date(2026, 1, 3), \"description\": \"Pie\", \"pence\": 350, \"currency\": \"GBP\"}]\n\ndef test_wrong_file():\n    \"the wrong kind of file\"\n    write_file(\"notes.csv\", \"name,score\\nAda,3\\n\")\n    try:\n        read_statement(\"notes.csv\")\n    except ValueError as e:\n        assert str(e) == \"notes.csv: expected the header date,description,amount,currency\"\n    else:\n        raise AssertionError(\"expected ValueError\")")

### Exercise: Step 2: exchange rates

Write `parse_rates(html)`: find the `<table id="rates">` and read each row whose two `<td>` cells are a three-capital-letter code and a positive decimal number (skip other rows, such as the `<th>` header). Return a dict from code to `Decimal`, always including `"GBP": Decimal("1")`; if the table holds no rates, raise `ValueError("no rates found on the page")`. Then `get_rates(fetch, cache_path, now, max_age=43200)`, where `now` is a time in seconds and `fetch(url)` returns `(status, body)` for `RATES_URL`. If the cache file exists and is less than `max_age` seconds old, return `(rates, "cache")` without fetching. Otherwise fetch: if the status isn't `200`, or `fetch` raises `OSError`, or `parse_rates` raises `ValueError`, return `(rates_from_the_cache, "stale cache")` if there's a cache, else raise `RuntimeError("no exchange rates available")`. After a good fetch, write the cache atomically (with the given `write_atomic`) as JSON `{"fetched_at": now, "rates": {code: str(rate)}}`, and return `(rates, "fetched")`.

In [ ]:
import json
import os
import re
from decimal import Decimal, InvalidOperation
from html.parser import HTMLParser
from pathlib import Path

RATES_URL = "https://rates.example/today"

def write_atomic(path, text):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)

def parse_rates(html):
    ___

def get_rates(fetch, cache_path, now, max_age=43200):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parse():\n    \"the rates table\"\n    from decimal import Decimal\n    rates = parse_rates(read_file(\"rates.html\"))\n    assert rates == {\"GBP\": Decimal(\"1\"), \"EUR\": Decimal(\"1.1500\"), \"USD\": Decimal(\"1.2500\"), \"JPY\": Decimal(\"190.20\")}\n    assert all(isinstance(v, Decimal) for v in rates.values())\n    odd = '<table id=\"other\"><tr><td>CHF</td><td>1.1</td></tr></table><table id=\"rates\"><tr><td>eur</td><td>1</td></tr><tr><td>NOK</td><td>-3</td></tr><tr><td>SEK</td><td>13.5</td></tr></table>'\n    assert parse_rates(odd) == {\"GBP\": Decimal(\"1\"), \"SEK\": Decimal(\"13.5\")}\n    try:\n        parse_rates(\"<p>Service unavailable</p>\")\n    except ValueError as e:\n        assert str(e) == \"no rates found on the page\"\n    else:\n        raise AssertionError(\"no table\")\n\ndef test_cache():\n    \"fetched once, then cached, then refreshed\"\n    import json\n    from decimal import Decimal\n    from pathlib import Path\n    Path(\"cache.json\").unlink(missing_ok=True)\n    calls = []\n    def fetch(url):\n        calls.append(url)\n        return 200, read_file(\"rates.html\")\n    rates, source = get_rates(fetch, \"cache.json\", 1000)\n    assert source == \"fetched\" and rates[\"EUR\"] == Decimal(\"1.1500\") and calls == [\"https://rates.example/today\"]\n    stored = json.loads(Path(\"cache.json\").read_text())\n    assert stored[\"fetched_at\"] == 1000 and stored[\"rates\"][\"USD\"] == \"1.2500\"\n    assert get_rates(fetch, \"cache.json\", 1000 + 43199)[1] == \"cache\" and len(calls) == 1\n    assert get_rates(fetch, \"cache.json\", 1000 + 43200)[1] == \"fetched\" and len(calls) == 2\n    assert not Path(\"cache.json.tmp\").exists()\n\ndef test_failures():\n    \"a stale cache beats no rates; no cache is an error\"\n    from decimal import Decimal\n    from pathlib import Path\n    Path(\"cache.json\").unlink(missing_ok=True)\n    get_rates(lambda url: (200, read_file(\"rates.html\")), \"cache.json\", 0)\n    for broken in (lambda url: (503, \"busy\"), lambda url: (200, \"<p>redesigned!</p>\")):\n        rates, source = get_rates(broken, \"cache.json\", 10 ** 6)\n        assert source == \"stale cache\" and rates[\"JPY\"] == Decimal(\"190.20\")\n    def down(url):\n        raise ConnectionError(\"no route to host\")\n    assert get_rates(down, \"cache.json\", 10 ** 6)[1] == \"stale cache\"\n    Path(\"cache.json\").unlink()\n    try:\n        get_rates(down, \"cache.json\", 10 ** 6)\n    except RuntimeError as e:\n        assert str(e) == \"no exchange rates available\"\n    else:\n        raise AssertionError(\"no cache and no site\")")

### Exercise: Step 3: the report

Write `to_gbp(pence, currency, rates)`: the amount in pence sterling, `pence / rate` rounded to a whole number with `ROUND_HALF_UP` (raise `KeyError(f"no rate for {currency}")` for an unknown currency). `category(description, rules=RULES)`: the category of the first rule whose keyword is in the lower-case description, or `"Other"`. `summarise(rows, rates, rules=RULES)`: `{"YYYY-MM": {category: total_pence}}`. `money(pence)`: pounds with a thousands comma and two decimals, and a minus sign for negatives (`-123456` gives `"-1,234.56"`). And `report(summary)`: for each month in order, the month on its own line, then one line per category in alphabetical order, `"  "` + the category left-aligned in 14 characters + the amount right-aligned in 10, then a `Total` line the same way; lines end with a newline. An empty summary gives `"nothing to report\n"`.

In [ ]:
from decimal import Decimal, ROUND_HALF_UP

RULES = [("tesco", "Groceries"), ("sainsbury", "Groceries"), ("rent", "Housing"), ("salary", "Income"),
         ("netflix", "Subscriptions"), ("cafe", "Eating out"), ("amazon", "Shopping")]

def to_gbp(pence, currency, rates):
    ___

def category(description, rules=RULES):
    ___

def summarise(rows, rates, rules=RULES):
    ___

def money(pence):
    ___

def report(summary):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _rates():\n    from decimal import Decimal\n    return {\"GBP\": Decimal(\"1\"), \"EUR\": Decimal(\"1.1500\"), \"USD\": Decimal(\"1.2500\")}\n\ndef test_convert():\n    \"exact conversion and rounding\"\n    rates = _rates()\n    assert to_gbp(-1150, \"EUR\", rates) == -1000 and to_gbp(-2500, \"USD\", rates) == -2000 and to_gbp(999, \"GBP\", rates) == 999\n    assert to_gbp(-1000, \"EUR\", rates) == -870 and to_gbp(1000, \"EUR\", rates) == 870 and to_gbp(5, \"USD\", rates) == 4\n    assert to_gbp(-5, \"USD\", rates) == -4 and to_gbp(-15, \"EUR\", rates) == -13\n    try:\n        to_gbp(100, \"CHF\", rates)\n    except KeyError:\n        pass\n    else:\n        raise AssertionError(\"unknown currency\")\n\ndef test_categories_and_money():\n    \"rules and formatting\"\n    assert category(\"TESCO STORES 2207\") == \"Groceries\" and category(\"Rent October\") == \"Housing\" and category(\"Bus\") == \"Other\"\n    assert category(\"Netflix\", [(\"net\", \"A\"), (\"netflix\", \"B\")]) == \"A\"\n    assert money(-123456) == \"-1,234.56\" and money(5) == \"0.05\" and money(-7) == \"-0.07\" and money(210000) == \"2,100.00\" and money(0) == \"0.00\"\n\ndef test_report():\n    \"the whole report\"\n    from datetime import date\n    rows = [{\"date\": date(2026, 10, 3), \"description\": \"TESCO STORES 2207\", \"pence\": -2340, \"currency\": \"GBP\"},\n            {\"date\": date(2026, 10, 5), \"description\": \"Salary ACME LTD\", \"pence\": 210000, \"currency\": \"GBP\"},\n            {\"date\": date(2026, 10, 12), \"description\": \"Cafe de Paris\", \"pence\": -1150, \"currency\": \"EUR\"},\n            {\"date\": date(2026, 11, 1), \"description\": \"Amazon Marketplace\", \"pence\": -2500, \"currency\": \"USD\"},\n            {\"date\": date(2026, 10, 30), \"description\": \"Tesco Extra\", \"pence\": -1000, \"currency\": \"GBP\"}]\n    s = summarise(rows, _rates())\n    assert s == {\"2026-10\": {\"Groceries\": -3340, \"Income\": 210000, \"Eating out\": -1000}, \"2026-11\": {\"Shopping\": -2000}}\n    assert report(s) == (\"2026-10\\n  Eating out        -10.00\\n  Groceries         -33.40\\n  Income          2,100.00\\n\"\n                         \"  Total           2,056.60\\n2026-11\\n  Shopping          -20.00\\n  Total             -20.00\\n\")\n    assert report({}) == \"nothing to report\\n\"")

### Exercise: Step 4: archiving safely

Write `archive_file(zip_path, path)`: add the file at `path` to the zip archive `zip_path` (mode `"a"`, so it's created if missing and added to if not, compressed with `ZIP_DEFLATED`, creating the archive's folder if needed), under its own name, or `STEM (1).EXT`, `STEM (2).EXT`... if that name is taken in the archive; return the name used. Then `process_inbox(inbox, archive_dir, state_path, month, handle)`: go through the inbox's `*.csv` files in sorted order. Load the state (`{"done": {}}` if there's no file). A file whose SHA-256 is in `state["done"]` is deleted and logged as `duplicate NAME (already processed as WHERE)`. Otherwise call `handle(path)`; if it raises `ValueError`, log `skipped NAME: MESSAGE` and leave the file where it is. Otherwise archive it into `archive_dir/MONTH.zip`, set `state["done"][hash] = "MONTH.zip:STORED_NAME"`, write the state atomically (as `json.dumps(state, indent=2, sort_keys=True)`), delete the file, and log `processed NAME -> MONTH.zip:STORED_NAME`. Return the log.

In [ ]:
import hashlib
import json
import os
import zipfile
from pathlib import Path

def write_atomic(path, text):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def load_state(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else {"done": {}}

def archive_file(zip_path, path):
    ___

def process_inbox(inbox, archive_dir, state_path, month, handle):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _setup():\n    import shutil\n    from pathlib import Path\n    shutil.rmtree(\"box\", ignore_errors=True)\n    Path(\"box/inbox\").mkdir(parents=True)\n    Path(\"box/inbox/a.csv\").write_text(\"date,description,amount,currency\\n2026-01-01,x,1,GBP\\n\")\n    Path(\"box/inbox/b.csv\").write_text(\"date,description,amount,currency\\n2026-01-02,y,2,GBP\\n\")\n    Path(\"box/inbox/junk.csv\").write_text(\"not a statement\\n\")\n    Path(\"box/inbox/readme.txt\").write_text(\"ignore me\")\n\ndef _handle(path):\n    if \"not a statement\" in path.read_text():\n        raise ValueError(\"expected the header\")\n\ndef test_process():\n    \"processed, archived, recorded, removed\"\n    import json, zipfile\n    from pathlib import Path\n    _setup()\n    log = process_inbox(\"box/inbox\", \"box/archive\", \"box/state.json\", \"2026-11\", _handle)\n    assert log == [\"processed a.csv -> 2026-11.zip:a.csv\", \"processed b.csv -> 2026-11.zip:b.csv\", \"skipped junk.csv: expected the header\"]\n    assert sorted(p.name for p in Path(\"box/inbox\").iterdir()) == [\"junk.csv\", \"readme.txt\"]\n    with zipfile.ZipFile(\"box/archive/2026-11.zip\") as zf:\n        assert zf.namelist() == [\"a.csv\", \"b.csv\"] and zf.read(\"b.csv\").startswith(b\"date,\")\n        assert zf.getinfo(\"a.csv\").compress_type == zipfile.ZIP_DEFLATED\n    state = json.loads(Path(\"box/state.json\").read_text())\n    assert sorted(state[\"done\"].values()) == [\"2026-11.zip:a.csv\", \"2026-11.zip:b.csv\"]\n\ndef test_idempotent():\n    \"duplicates are recognised by content, and names never clash\"\n    import zipfile\n    from pathlib import Path\n    _setup()\n    process_inbox(\"box/inbox\", \"box/archive\", \"box/state.json\", \"2026-11\", _handle)\n    Path(\"box/inbox/a-copy.csv\").write_text(\"date,description,amount,currency\\n2026-01-01,x,1,GBP\\n\")\n    Path(\"box/inbox/a.csv\").write_text(\"date,description,amount,currency\\n2026-02-01,new,5,GBP\\n\")\n    log = process_inbox(\"box/inbox\", \"box/archive\", \"box/state.json\", \"2026-11\", _handle)\n    assert log == [\"duplicate a-copy.csv (already processed as 2026-11.zip:a.csv)\", \"processed a.csv -> 2026-11.zip:a (1).csv\",\n                   \"skipped junk.csv: expected the header\"]\n    with zipfile.ZipFile(\"box/archive/2026-11.zip\") as zf:\n        assert zf.namelist() == [\"a.csv\", \"b.csv\", \"a (1).csv\"]\n    assert process_inbox(\"box/inbox\", \"box/archive\", \"box/state.json\", \"2026-12\", _handle) == [\"skipped junk.csv: expected the header\"]\n    assert not Path(\"box/archive/2026-12.zip\").exists()")

### Exercise: Step 5: when's the next run?

Write `next_run(now, hour=2, minute=30)`: the next `datetime` strictly after `now` at that hour and minute (with zero seconds). Then `status_lines(new_files, done_count, fetched_at, now)`, which returns the lines `ledger status` prints: `inbox: N new file(s)`, `processed: M file(s)`, then `rates: fetched YYYY-MM-DD HH:MM` (where `fetched_at` is seconds since 1970-01-01 00:00, turned into a `datetime` by adding a `timedelta` to `datetime(1970, 1, 1)`) or `rates: none yet` when `fetched_at` is `None`, and `next run: YYYY-MM-DD HH:MM`.

In [ ]:
from datetime import datetime, timedelta

EPOCH = datetime(1970, 1, 1)

def next_run(now, hour=2, minute=30):
    ___

def status_lines(new_files, done_count, fetched_at, now):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_next_run():\n    \"strictly after now\"\n    from datetime import datetime\n    assert next_run(datetime(2026, 11, 10, 1, 0)) == datetime(2026, 11, 10, 2, 30)\n    assert next_run(datetime(2026, 11, 10, 2, 30)) == datetime(2026, 11, 11, 2, 30)\n    assert next_run(datetime(2026, 12, 31, 23, 59, 59)) == datetime(2027, 1, 1, 2, 30)\n    assert next_run(datetime(2026, 11, 10, 7, 0), 6, 15) == datetime(2026, 11, 11, 6, 15)\n\ndef test_status():\n    \"the status report\"\n    from datetime import datetime\n    now = datetime(2026, 11, 10, 9, 41, 12)\n    assert status_lines(2, 0, None, now) == [\"inbox: 2 new file(s)\", \"processed: 0 file(s)\", \"rates: none yet\", \"next run: 2026-11-11 02:30\"]\n    assert status_lines(0, 5, 1794277800.0, now)[2] == \"rates: fetched 2026-11-10 02:30\"")

### Bug hunt: A penny out

The totals are kept as floats, so tiny binary errors creep in and the check fails: this prints `0.30000000000000004 False`. Keep money exact (add `Decimal`s made from the strings, or whole pence), so it prints `0.30 True`.

In [ ]:
amounts = ["0.10", "0.20"]
total = sum(float(a) for a in amounts)
print(total, total == 0.3)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"Decimal\"\n    expect_output(\"0.30 True\")")

### Bug hunt: Rounding towards zero

`int()` drops the fraction instead of rounding, so `-869.57` pence becomes `-869` and `869.57` becomes `869`, and this prints `-869 869`. Round to the nearest penny, halves away from zero, so it prints `-870 870`.

In [ ]:
from decimal import Decimal

rate = Decimal("1.15")
print(int(Decimal(-1000) / rate), int(Decimal(1000) / rate))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"ROUND_HALF_UP\"\n    expect_output(\"-870 870\")")

### Bug hunt: A cache that was never used

The freshness check is the wrong way round: it uses the cache only when it's **older** than the limit, so a fresh cache is ignored and the site is fetched every time. This prints `fetched fetched fetched`. Fix the comparison, so the second call uses the cache and only the third (13 hours later) fetches again: `fetched cache fetched`.

In [ ]:
MAX_AGE = 12 * 3600
cache = None

def get(now):
    global cache
    if cache is not None and now - cache > MAX_AGE:
        return "cache"
    cache = now
    return "fetched"

print(get(0), get(3600), get(13 * 3600))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fresh means younger than the limit\"\n    expect_output(\"fetched cache fetched\")")

### Puzzle: One statement, safely

Arrange the handling of one statement so that a crash at any point never loses it or counts it twice. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # int(Decimal("-12.345") * 100)
q = ___   # Decimal("2.5").quantize(Decimal("1"), rounding=ROUND_HALF_UP), as an int
r = ___   # what identifies a statement as already processed: "name" or "contents"
s = ___   # the exit code of a run that finished but found bad rows
t = ___   # True or False: a crash after archiving but before saving the state loses the statement

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == -1234 and q == 3 and r == \"contents\", \"p, q and r\"\n    assert s == 1 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: ledger

Write `main(argv, root, fetch, now, out, err)` (all the steps' functions are given). `root` is the ledger folder, `fetch(url)` returns `(status, body)`, `now` is a `datetime`, and output goes to `out` and `err`. Usage errors print `ledger: MESSAGE` to `err` and return `2` (nothing may exit). **`status`** prints `status_lines(...)` (new files are inbox CSVs whose hash isn't in the state; `fetched_at` comes from `rates.json` if it exists) and returns `0`. **`run [--dry-run]`**: take the lock `root/ledger.lock` (open with `"x"`; if it exists print `ledger: another run is in progress` to `err` and return `1`), and always remove it at the end. Get the rates into the cache `root/rates.json` with `now` as seconds since `EPOCH` (on `RuntimeError`, print `ledger: no exchange rates available` to `err` and return `1`). Read each new statement with `read_statement`; rows whose currency has no rate are dropped with the warning `NAME: no rate for CODE`, and row errors are warnings too. In a **dry run**, for each inbox CSV in order print `would skip duplicate NAME`, `would skip NAME: MESSAGE` (for a bad header) or `would process NAME (N rows)` (counting the rows `read_statement` accepted), then write the report text to `out`, then `dry run: nothing changed`; nothing else changes (the rates cache may be updated). Otherwise run `process_inbox` with this month (`YYYY-MM` of `now`) and print its log lines; then, if any rows were read, write the report to `root/report.txt` atomically and print `report: N row(s) in M month(s); rates: SOURCE`, else print `nothing new to report`. Warnings are printed last, to `err`, as `ledger: warning: TEXT`. Return `1` if there were warnings, else `0`.

In [ ]:
import argparse
import csv
import hashlib
import json
import os
import re
import zipfile
from datetime import date, datetime, timedelta
from decimal import Decimal, ROUND_HALF_UP, InvalidOperation
from html.parser import HTMLParser
from pathlib import Path

HEADER = ["date", "description", "amount", "currency"]
RATES_URL = "https://rates.example/today"
RULES = [("tesco", "Groceries"), ("sainsbury", "Groceries"), ("rent", "Housing"), ("salary", "Income"),
         ("netflix", "Subscriptions"), ("cafe", "Eating out"), ("amazon", "Shopping")]
EPOCH = datetime(1970, 1, 1)

def read_statement(path):
    path = Path(path)
    rows, errors = [], []
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        if next(reader, None) != HEADER:
            raise ValueError(f"{path.name}: expected the header {','.join(HEADER)}")
        for line, record in enumerate(reader, 2):
            if not record:
                continue
            if len(record) != 4:
                errors.append(f"{path.name}:{line}: expected 4 fields")
                continue
            day, description, amount, currency = (x.strip() for x in record)
            try:
                when = date.fromisoformat(day)
            except ValueError:
                errors.append(f"{path.name}:{line}: bad date")
                continue
            try:
                value = Decimal(amount)
                if value != value.quantize(Decimal("0.01")):
                    raise InvalidOperation
            except InvalidOperation:
                errors.append(f"{path.name}:{line}: bad amount")
                continue
            if not re.fullmatch(r"[A-Z]{3}", currency):
                errors.append(f"{path.name}:{line}: bad currency")
                continue
            rows.append({"date": when, "description": description, "pence": int(value * 100), "currency": currency})
    return rows, errors

class RateTable(HTMLParser):
    def __init__(self):
        super().__init__()
        self.inside, self.cells, self.cell, self.rows = False, None, None, []
    def handle_starttag(self, tag, attrs):
        if tag == "table" and dict(attrs).get("id") == "rates":
            self.inside = True
        elif self.inside and tag == "tr":
            self.cells = []
            self.rows.append(self.cells)
        elif self.inside and tag == "td":
            self.cell = ""
    def handle_endtag(self, tag):
        if tag == "table":
            self.inside = False
        elif tag == "td" and self.cell is not None:
            self.cells.append(self.cell.strip())
            self.cell = None
    def handle_data(self, data):
        if self.cell is not None:
            self.cell += data

def parse_rates(html):
    p = RateTable()
    p.feed(html)
    p.close()
    rates = {"GBP": Decimal("1")}
    for cells in p.rows:
        if len(cells) == 2 and re.fullmatch(r"[A-Z]{3}", cells[0]):
            try:
                rate = Decimal(cells[1])
            except InvalidOperation:
                continue
            if rate > 0:
                rates[cells[0]] = rate
    if len(rates) == 1:
        raise ValueError("no rates found on the page")
    return rates

def write_atomic(path, text):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)

def get_rates(fetch, cache_path, now, max_age=43200):
    cache_path = Path(cache_path)
    cached = json.loads(cache_path.read_text()) if cache_path.exists() else None
    if cached and now - cached["fetched_at"] < max_age:
        return {k: Decimal(v) for k, v in cached["rates"].items()}, "cache"
    try:
        status, body = fetch(RATES_URL)
        if status != 200:
            raise OSError(f"status {status}")
        rates = parse_rates(body)
    except (OSError, ValueError):
        if cached:
            return {k: Decimal(v) for k, v in cached["rates"].items()}, "stale cache"
        raise RuntimeError("no exchange rates available") from None
    write_atomic(cache_path, json.dumps({"fetched_at": now, "rates": {k: str(v) for k, v in rates.items()}}))
    return rates, "fetched"

def to_gbp(pence, currency, rates):
    if currency not in rates:
        raise KeyError(f"no rate for {currency}")
    return int((Decimal(pence) / rates[currency]).quantize(Decimal("1"), rounding=ROUND_HALF_UP))

def category(description, rules=RULES):
    text = description.lower()
    return next((cat for word, cat in rules if word in text), "Other")

def summarise(rows, rates, rules=RULES):
    summary = {}
    for row in rows:
        month = summary.setdefault(row["date"].strftime("%Y-%m"), {})
        cat = category(row["description"], rules)
        month[cat] = month.get(cat, 0) + to_gbp(row["pence"], row["currency"], rates)
    return summary

def money(pence):
    sign = "-" if pence < 0 else ""
    return f"{sign}{abs(pence) // 100:,}.{abs(pence) % 100:02d}"

def report(summary):
    lines = []
    for month in sorted(summary):
        lines.append(month)
        for cat in sorted(summary[month]):
            lines.append(f"  {cat:<14}{money(summary[month][cat]):>10}")
        lines.append(f"  {'Total':<14}{money(sum(summary[month].values())):>10}")
    return "\n".join(lines) + "\n" if lines else "nothing to report\n"

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def load_state(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else {"done": {}}

def archive_file(zip_path, path):
    zip_path = Path(zip_path)
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "a", compression=zipfile.ZIP_DEFLATED) as zf:
        names = set(zf.namelist())
        p = Path(path)
        name, n = p.name, 0
        while name in names:
            n += 1
            name = f"{p.stem} ({n}){p.suffix}"
        zf.write(p, arcname=name)
    return name

def process_inbox(inbox, archive_dir, state_path, month, handle):
    inbox, archive_dir = Path(inbox), Path(archive_dir)
    state = load_state(state_path)
    log = []
    for path in sorted(inbox.glob("*.csv")):
        digest = file_hash(path)
        if digest in state["done"]:
            path.unlink()
            log.append(f"duplicate {path.name} (already processed as {state['done'][digest]})")
            continue
        try:
            handle(path)
        except ValueError as e:
            log.append(f"skipped {path.name}: {e}")
            continue
        stored = archive_file(archive_dir / f"{month}.zip", path)
        state["done"][digest] = f"{month}.zip:{stored}"
        write_atomic(state_path, json.dumps(state, indent=2, sort_keys=True))
        path.unlink()
        log.append(f"processed {path.name} -> {month}.zip:{stored}")
    return log

def next_run(now, hour=2, minute=30):
    run = now.replace(hour=hour, minute=minute, second=0, microsecond=0)
    return run if run > now else run + timedelta(days=1)

def status_lines(new_files, done_count, fetched_at, now):
    lines = [f"inbox: {new_files} new file(s)", f"processed: {done_count} file(s)"]
    if fetched_at is None:
        lines.append("rates: none yet")
    else:
        lines.append(f"rates: fetched {EPOCH + timedelta(seconds=fetched_at):%Y-%m-%d %H:%M}")
    lines.append(f"next run: {next_run(now):%Y-%m-%d %H:%M}")
    return lines

def main(argv, root, fetch, now, out, err):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _fresh():\n    import shutil\n    from pathlib import Path\n    shutil.rmtree(\"ledger\", ignore_errors=True)\n    Path(\"ledger/inbox\").mkdir(parents=True)\n    for name in (\"2026-10-bank.csv\", \"2026-11-card.csv\"):\n        Path(\"ledger/inbox\", name).write_text(read_file(\"inbox/\" + name))\n\ndef _site(status=200):\n    calls = []\n    def fetch(url):\n        calls.append(url)\n        return status, read_file(\"rates.html\")\n    return fetch, calls\n\ndef _run(argv, fetch, now=None):\n    import io\n    from datetime import datetime\n    out, err = io.StringIO(), io.StringIO()\n    code = main(argv, \"ledger\", fetch, now or datetime(2026, 11, 10, 2, 30), out, err)\n    return code, out.getvalue(), err.getvalue()\n\nREPORT = \"\"\"2026-10\n  Eating out        -10.00\n  Groceries         -23.40\n  Housing          -850.00\n  Income          2,100.00\n  Total           1,216.60\n2026-11\n  Groceries         -41.10\n  Shopping          -20.00\n  Subscriptions     -10.99\n  Total             -72.09\n\"\"\"\nWARNINGS = (\"ledger: warning: 2026-11-card.csv:4: bad amount\\nledger: warning: 2026-11-card.csv:5: bad date\\n\"\n            \"ledger: warning: 2026-11-card.csv:7: bad currency\\n\")\n\ndef test_dry_run():\n    \"a dry run changes nothing\"\n    from pathlib import Path\n    _fresh()\n    fetch, calls = _site()\n    code, out, err = _run([\"run\", \"--dry-run\"], fetch)\n    assert code == 1 and err == WARNINGS\n    assert out == \"would process 2026-10-bank.csv (4 rows)\\nwould process 2026-11-card.csv (3 rows)\\n\" + REPORT + \"dry run: nothing changed\\n\"\n    assert sorted(p.name for p in Path(\"ledger/inbox\").iterdir()) == [\"2026-10-bank.csv\", \"2026-11-card.csv\"]\n    assert not Path(\"ledger/report.txt\").exists() and not Path(\"ledger/state.json\").exists() and not Path(\"ledger/ledger.lock\").exists()\n\ndef test_run_twice():\n    \"a real run, then a run with nothing new\"\n    import zipfile\n    from pathlib import Path\n    _fresh()\n    fetch, calls = _site()\n    code, out, err = _run([\"run\"], fetch)\n    assert code == 1 and err == WARNINGS\n    assert out == (\"processed 2026-10-bank.csv -> 2026-11.zip:2026-10-bank.csv\\nprocessed 2026-11-card.csv -> 2026-11.zip:2026-11-card.csv\\n\"\n                   \"report: 7 row(s) in 2 month(s); rates: fetched\\n\")\n    assert Path(\"ledger/report.txt\").read_text() == REPORT and list(Path(\"ledger/inbox\").iterdir()) == []\n    assert zipfile.ZipFile(\"ledger/archive/2026-11.zip\").namelist() == [\"2026-10-bank.csv\", \"2026-11-card.csv\"]\n    code, out, err = _run([\"run\"], fetch)\n    assert (code, out, err) == (0, \"nothing new to report\\n\", \"\") and len(calls) == 1\n    assert Path(\"ledger/report.txt\").read_text() == REPORT and not Path(\"ledger/ledger.lock\").exists()\n\ndef test_status():\n    \"status before and after\"\n    _fresh()\n    fetch, calls = _site()\n    assert _run([\"status\"], fetch) == (0, \"inbox: 2 new file(s)\\nprocessed: 0 file(s)\\nrates: none yet\\nnext run: 2026-11-11 02:30\\n\", \"\")\n    _run([\"run\"], fetch)\n    assert _run([\"status\"], fetch)[1] == \"inbox: 0 new file(s)\\nprocessed: 2 file(s)\\nrates: fetched 2026-11-10 02:30\\nnext run: 2026-11-11 02:30\\n\"\n    assert calls == [\"https://rates.example/today\"]\n\ndef test_problems():\n    \"locks, missing rates, unknown currencies and usage\"\n    from pathlib import Path\n    _fresh()\n    Path(\"ledger/ledger.lock\").write_text(\"\")\n    assert _run([\"run\"], _site()[0]) == (1, \"\", \"ledger: another run is in progress\\n\")\n    assert Path(\"ledger/ledger.lock\").exists()\n    Path(\"ledger/ledger.lock\").unlink()\n    assert _run([\"run\"], _site(503)[0]) == (1, \"\", \"ledger: no exchange rates available\\n\")\n    assert not Path(\"ledger/ledger.lock\").exists() and len(list(Path(\"ledger/inbox\").iterdir())) == 2\n    _fresh()\n    Path(\"ledger/inbox/2026-10-bank.csv\").unlink()\n    Path(\"ledger/inbox/2026-11-card.csv\").unlink()\n    Path(\"ledger/inbox/swiss.csv\").write_text(\"date,description,amount,currency\\n2026-11-03,Chocolate,-9.00,CHF\\n2026-11-03,TESCO,-1.00,GBP\\n\")\n    Path(\"ledger/inbox/wrong.csv\").write_text(\"hello\\n\")\n    code, out, err = _run([\"run\"], _site()[0])\n    assert code == 1 and err == \"ledger: warning: swiss.csv: no rate for CHF\\n\"\n    assert out == \"processed swiss.csv -> 2026-11.zip:swiss.csv\\nskipped wrong.csv: wrong.csv: expected the header date,description,amount,currency\\nreport: 1 row(s) in 1 month(s); rates: fetched\\n\"\n    for argv in ([], [\"fly\"], [\"run\", \"--now\"]):\n        code, out, err = _run(argv, _site()[0])\n        assert code == 2 and err.startswith(\"ledger: \"), argv")

## The plan

Money is never a `float`: binary floating point can't hold `0.1` exactly, and the errors add up across thousands of rows. The tool keeps every amount as a whole number of **pence**, uses `Decimal` for conversions, and rounds **once**, deliberately. Note that `int()` simply drops the fraction (towards zero), which isn't the same as rounding.

Here's what ledger works with. The page provides a sample: an inbox with two statements and a saved copy of the exchange-rate page (open them in the files panel).

The finished tool keeps everything in one folder:

```
ledger/
  inbox/          statements waiting to be processed
  archive/        2026-11.zip, ... (processed statements, by the month they were processed)
  state.json      the SHA-256 of every statement already processed
  rates.json      the cached exchange rates, with when they were fetched
  report.txt      the latest report
  ledger.lock     present only while a run is going
```

and it's run as `ledger run` (or `ledger run --dry-run` to see what would happen) and `ledger status`. Five steps build it, then the daily challenge puts them together.

## Step 1: reading statements

Statements come from outside, so every row is **untrusted** (Week 23). The rules: the first line must be exactly the header `date,description,amount,currency`; a bad header means it's the wrong kind of file, so refuse the whole file. After that, check each row and **collect** problems rather than stopping at the first, so one run reports everything that needs fixing: a wrong number of fields, a date that isn't a real ISO date, an amount that isn't a decimal number with at most two decimal places, a currency that isn't three capital letters. Report each with the file name and line number (the header is line 1), the way compilers do: `2026-11-card.csv:4: bad amount`.

## Step 2: exchange rates

The rates come from a web page, a table with the id `rates` listing how many units of each currency one pound buys. Parse it with `HTMLParser` (Thursday), always adding `GBP` at `1`. Keep rates as `Decimal`s, made from the cell's **text** (`Decimal("1.1500")`), never from a float.

Fetching it every run would be rude and fragile, so **cache** it: store the rates and the time they were fetched in `rates.json`, and use the cache while it's fresh (12 hours). When it's old, fetch again; if the fetch fails (the site is down, returns an error, or the page has no table any more), fall back to the **stale** cache rather than failing the whole night's run, and say so. Only with no cache at all does the run fail. The cache is written **atomically** (Tuesday), so a crash can't leave a half-written file that breaks every later run.

## Step 3: the report

Each row's amount is in its own currency's minor units (pence, cents). To convert into pence sterling, divide by the rate (one pound buys `1.15` euros, so `-1150` cents is `-1000` pence) and round to a whole penny, **half away from zero** (`ROUND_HALF_UP` in `Decimal`'s terms), in one place, so every report rounds the same way.

Each row also gets a **category** from its description, by simple rules: the first rule whose keyword appears in the lower-case description wins (`"tesco"` is Groceries), and anything unmatched is `Other`. The report groups the totals by month, then category:

```
2026-10
  Eating out        -10.00
  Groceries         -23.40
  Housing          -850.00
  Income          2,100.00
  Total           1,216.60
```

## Step 4: archiving safely

The night's run must be **idempotent**: if it runs twice, or crashes halfway and runs again, no statement is counted twice and none is lost. The design:

- Identify each statement by the **SHA-256 of its contents**, not its name: the bank names every export `statement.csv`, and a renamed copy is still the same statement.
- Keep the hashes of everything processed in `state.json`, written atomically after **each** file.
- For each statement in the inbox, in order: if its hash is already in the state, it's a duplicate: delete it and say so. Otherwise **process** it, **archive** it into this month's zip (with a free name inside the zip if that name is taken), **record** it in the state, and only then **delete** it from the inbox.

Think about a crash at each point. Before the archive: the file is still in the inbox, and the next run processes it. After archiving but before recording: it's archived twice next time (harmless; nothing is lost and the report isn't affected, since the state decides). After recording but before deleting: next time it's recognised as a duplicate and deleted. No crash anywhere loses a statement or counts one twice in the state. That ordering is the whole design.

## Step 5: every night

The last step wraps the parts in a command-line tool (Monday) that's safe to run unattended (Wednesday):

- `ledger run` takes a **lock** first (a second copy started while the first is still going must not process the same files), gets the rates, processes the inbox, writes the report atomically, and releases the lock in a `finally`;
- `ledger run --dry-run` reads and reports, but writes, archives and deletes nothing, so you can see what a run would do;
- `ledger status` says what's waiting, what's been done, how old the rates are, and when the next run is;
- results go to **stdout**; problems in the data go to **stderr** as warnings; the exit code is `0` when all went well, `1` when there were warnings or the run couldn't happen (so cron's mail, or a monitor, tells you), and `2` for a usage mistake;
- everything it depends on (the folder, the fetch function, the time, the streams) is a parameter of `main`, which is why every behaviour can be tested in a second.

On a Linux server, one crontab line runs it at 02:30 every night, with absolute paths and the output kept in a log:

```
30 2 * * * /home/ada/ledger/.venv/bin/python -m ledger run >> /home/ada/ledger/ledger.log 2>&1
```

## Checkpoint and quiz

## Daily challenge

## Before you go

Think about one dull, regular job in your own life or work (a weekly report, renaming downloads, checking a page for changes) and sketch the steps it would need: what it reads, what it checks, what it writes, and what must never happen twice. Tomorrow is the Week 78 **Forge**.